# Nova AI — Production GPU Runner
Uses the existing Nova training pipeline; no rebuild and no credentials printed.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

TARGET=50_000_000

def find_root():
    for r in [Path('/content/NOVA_AUDIT'),Path('/content/nova-v11'),Path('/content')]:
        if (r/'autopilot.py').exists(): return r
    hits=list(Path('/content').rglob('autopilot.py'))
    if hits: return hits[0].parent
    raise FileNotFoundError('Nova autopilot.py not found in /content')

root=find_root(); os.chdir(root)
import torch
assert torch.cuda.is_available(), 'NVIDIA GPU/CUDA is not available'
props=torch.cuda.get_device_properties(0)
x=torch.randn(1024,1024,device='cuda'); _=x@x; torch.cuda.synchronize()
print({'gpu':torch.cuda.get_device_name(0),'vram_gb':round(props.total_memory/1024**3,2),'cuda':torch.version.cuda})

drive=None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive=Path('/content/drive/MyDrive/NOVA_CHECKPOINTS'); drive.mkdir(parents=True,exist_ok=True)
except Exception as e:
    print('Drive persistence unavailable:',e)

checkpoint=str((drive/'nova-stream.pt') if drive else (root/'checkpoints/nova-stream.pt'))
Path(checkpoint).parent.mkdir(parents=True,exist_ok=True)
local=root/'checkpoints/nova-stream.pt'
if drive and (drive/'nova-stream.pt').exists() and not local.exists():
    local.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(drive/'nova-stream.pt',local)
      
for attempt in range(1,4):
    print(f'=== Nova production pass {attempt}/3 ===')
    code=subprocess.call([sys.executable,'autopilot.py','--pretrain-tokens',str(TARGET),'--do-posttrain','--prepare-posttrain','--checkpoint',checkpoint],env={**os.environ,'PYTHONUNBUFFERED':'1'})
    finals=[root/'checkpoints/nova-final.pt',root/'production/final.pt',root/'artifacts/final.pt',root/'checkpoints/nova-preference.pt']
    final=next((p for p in finals if p.exists() and p.stat().st_size>0),None)
    if final:
        if drive:
            for src in {final, final.parent/'nova-stream.pt',final.parent/'nova-preference.pt',final.parent/'nova-instruct.pt'}:
                if src.exists() and src.is_file(): shutil.copy2(src,drive/src.name)
        print(json.dumps({'ok':True,'status':'complete','target_tokens':TARGET,'final_checkpoint':str(final),'size_bytes':final.stat().st_size},indent=2))
        break
    print('Autopilot pass ended:',code)
    if attempt==3: raise RuntimeError('No verified final checkpoint produced')
    time.sleep(10)
